# MRI Stroke Assist -- Improved 3D U-Net Training (v3)

Training notebook for Kaggle GPU environment.

**Previous results:**
- Baseline (no aug): dice=0.606 at epoch 92
- v1 (aggressive aug): dice=0.405 -- RandAffined killed performance
- v2 (mild aug): dice=0.438 -- still too much augmentation

**v3 strategy: minimal augmentation**
1. Only RandFlipd (safest, doubles effective dataset) + very light noise (std=0.03)
2. No rotation, no scaling, no gamma -- these hurt on small dataset
3. Batch size 2 -> 4, dropout stays 0.1
4. Epochs 150, patience 30

**Setup:**
1. Add dataset: `orvile/isles-2022-brain-stoke-dataset`
2. Enable GPU: Settings -> Accelerator -> GPU T4 x2
3. Run all cells

## 1. Setup environment

In [ ]:
# Clone our repo
!git clone https://gitlab.com/Payz111/mri-stroke-assistance.git /kaggle/working/mri-stroke-assist
%cd /kaggle/working/mri-stroke-assist

In [ ]:
# Install dependencies (most are pre-installed on Kaggle)
!pip install -q monai nibabel SimpleITK pyyaml

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Check data paths

Kaggle mounts the dataset at `/kaggle/input/isles-2022-brain-stoke-dataset/`.
We need to find the exact structure.

In [ ]:
import os
from pathlib import Path

# Find the dataset root
kaggle_input = Path("/kaggle/input/isles-2022-brain-stoke-dataset")
print("Dataset contents:")
for item in sorted(kaggle_input.iterdir()):
    print(f"  {item.name}/" if item.is_dir() else f"  {item.name}")

# Explore deeper to find ISLES-2022 structure
for root, dirs, files in os.walk(kaggle_input):
    depth = root.replace(str(kaggle_input), "").count(os.sep)
    if depth < 3:
        indent = " " * 2 * depth
        print(f"{indent}{os.path.basename(root)}/")
        if depth == 2:
            print(f"{indent}  ... ({len(dirs)} dirs, {len(files)} files)")
            break

In [ ]:
# Determine data_root and derivatives_root
# Adjust these paths based on the actual Kaggle dataset structure

# Try common structures
possible_roots = [
    kaggle_input / "ISLES-2022",
    kaggle_input,
    kaggle_input / "isles22" / "ISLES-2022",
]

data_root = None
for root in possible_roots:
    if (root / "sub-strokecase0001").exists():
        data_root = root
        break

if data_root is None:
    # Search for it
    for root, dirs, files in os.walk(kaggle_input):
        if "sub-strokecase0001" in dirs:
            data_root = Path(root)
            break

assert data_root is not None, "Could not find ISLES-2022 data root!"
derivatives_root = data_root / "derivatives"

print(f"data_root: {data_root}")
print(f"derivatives_root: {derivatives_root}")
print(f"derivatives exists: {derivatives_root.exists()}")

# Count subjects
subjects = sorted([d.name for d in data_root.iterdir() if d.name.startswith("sub-")])
print(f"Subjects found: {len(subjects)}")

## 3. Copy splits to working directory

Our splits are in the repo. No need to recreate them.

In [ ]:
import json

split_file = Path("/kaggle/working/mri-stroke-assist/data/splits/fold_0.json")
with open(split_file) as f:
    split = json.load(f)

print(f"Fold 0: {split['n_train']} train, {split['n_val']} val")

## 4. Test dataset loading

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/mri-stroke-assist")

from src.data.isles22_dataset import ISLES22Dataset
from src.data.transforms import get_train_transforms, get_val_transforms

# Quick test: load 1 sample
ds_test = ISLES22Dataset(
    data_root=data_root,
    derivatives_root=derivatives_root,
    split_file=split_file,
    split="train",
    transform=get_val_transforms(),
)

sample = ds_test[0]
print(f"image: {sample['image'].shape}, dtype={sample['image'].dtype}")
print(f"label: {sample['label'].shape}, dtype={sample['label'].dtype}")
print("Dataset loading OK!")

## 5. Train!

In [ ]:
import logging
import time
import yaml

from torch.utils.data import DataLoader
from src.models.factory import create_model, create_loss
from src.train.trainer import Trainer
from src.train.callbacks import CheckpointCallback, EarlyStoppingCallback

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s: %(message)s", datefmt="%H:%M:%S")

# --- Load all config from improved.yaml ---
with open("/kaggle/working/mri-stroke-assist/configs/experiment/improved.yaml") as f:
    improved_cfg = yaml.safe_load(f)

FOLD = 0
EPOCHS = improved_cfg["training"]["epochs"]           # 150
BATCH_SIZE = improved_cfg["training"]["batch_size"]    # 4
LR = improved_cfg["training"]["learning_rate"]         # 1e-4
DROPOUT = improved_cfg["model"]["dropout"]             # 0.1
PATIENCE = improved_cfg["training"]["early_stopping_patience"]  # 30
AUG_CONFIG = improved_cfg.get("augmentation", {})
NUM_WORKERS = 2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Config: epochs={EPOCHS}, batch={BATCH_SIZE}, lr={LR}, dropout={DROPOUT}, patience={PATIENCE}")
print(f"Augmentation: {AUG_CONFIG}")
print(f"Device: {DEVICE}")

In [ ]:
# Datasets -- now with augmentation!
train_ds = ISLES22Dataset(
    data_root=data_root,
    derivatives_root=derivatives_root,
    split_file=split_file,
    split="train",
    transform=get_train_transforms(AUG_CONFIG),  # augmentation applied!
)
val_ds = ISLES22Dataset(
    data_root=data_root,
    derivatives_root=derivatives_root,
    split_file=split_file,
    split="val",
    transform=get_val_transforms(),  # no augmentation for val
)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

# DataLoaders
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

In [ ]:
# Model
model_cfg = {"name": "unet3d", "in_channels": 3, "out_channels": 1, "features": [32, 64, 128, 256], "dropout": DROPOUT}
loss_cfg = {"type": "dice_focal", "dice_weight": 0.5, "focal_weight": 0.5, "focal_gamma": 2.0}

model = create_model(model_cfg)
criterion = create_loss(loss_cfg)

param_count = sum(p.numel() for p in model.parameters())
print(f"Model params: {param_count:,}, dropout={DROPOUT}")

# Optimizer & scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-7)

# Callbacks
output_dir = Path("/kaggle/working/outputs/fold_0")
output_dir.mkdir(parents=True, exist_ok=True)

callbacks = [
    CheckpointCallback(save_dir=output_dir / "checkpoints", monitor="val_dice"),
    EarlyStoppingCallback(patience=PATIENCE, monitor="val_dice"),
]
print(f"Early stopping patience: {PATIENCE}")

In [ ]:
# Train
trainer = Trainer(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    scheduler=scheduler,
    callbacks=callbacks,
)

t0 = time.time()
result = trainer.fit(num_epochs=EPOCHS)
elapsed = time.time() - t0

print(f"\nTraining complete in {elapsed/60:.1f} min")
print(f"Best val_dice: {result['best_val_dice']:.4f} at epoch {result['best_epoch'] + 1}")
print(f"\n--- Comparison ---")
print(f"Baseline (no aug):   0.6062 at epoch 92")
print(f"v1 (aggressive aug): 0.4045 at epoch 41")
print(f"v2 (mild aug):       0.4381 at epoch 32")
print(f"v3 (flip + noise):   {result['best_val_dice']:.4f} at epoch {result['best_epoch'] + 1}")
print(f"Change vs baseline:  {result['best_val_dice'] - 0.6062:+.4f}")

## 6. Training curves

In [ ]:
import matplotlib.pyplot as plt

history = result["history"]
epochs_range = [h["epoch"] + 1 for h in history]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(epochs_range, [h["train_loss"] for h in history], label="Train")
axes[0].plot(epochs_range, [h["val_loss"] for h in history], label="Val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss")
axes[0].legend()
axes[0].grid(True)

# Dice
axes[1].plot(epochs_range, [h["train_dice"] for h in history], label="Train")
axes[1].plot(epochs_range, [h["val_dice"] for h in history], label="Val")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Dice")
axes[1].set_title("Dice Score")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig("/kaggle/working/outputs/training_curves.png", dpi=150)
plt.show()
print(f"Best val Dice: {result['best_val_dice']:.4f}")

## 7. Save results

Download the checkpoint and training curves from `/kaggle/working/outputs/`.

In [ ]:
import json

# Save training history
history_path = output_dir / "training_history.json"
with open(history_path, "w") as f:
    json.dump(result["history"], f, indent=2)

print(f"Checkpoint: {output_dir / 'checkpoints' / 'best_model.pth'}")
print(f"History: {history_path}")
print(f"Curves: /kaggle/working/outputs/training_curves.png")
print("\nDownload these files from the Output tab!")